# Durable batch manifests and explicit payloads

SDK 0.5 candidate / API-049, API-050, API-051. This notebook runs a controlled synthetic transport. It executes no foundation model, SQL checkpoint workflow, cloud job, deletion or resource qualification. Batch is disabled in the current service and no native task is qualified. Explicit live mode refuses before a request.

The fixture deliberately returns one failed half-open range. These values illustrate row placement and client behavior, not scientific predictions or retry quality.

In [ ]:
from support import BatchSyntheticService, notebook_batch_client, prepare
from welt import BatchResult, BatchPayload, ResultExpiredError

service = BatchSyntheticService()
with notebook_batch_client(service) as client:
    columns, query, predictor = prepare(client)
    query_dataset = client.upload(columns=columns, rows=query[:5].tolist(),
                                  name="Synthetic batch query")
    job = client.submit_batch(query_dataset["id"], predictor_id=predictor["id"],
                              probabilities=True, idempotency_key="notebook-batch-v1")
    job_id = job.id
    result = job.result()
    assert isinstance(result, BatchResult)
    assert result.status == "partially_completed"
    assert result.successful_ranges == ((0, 2), (4, 5))
    assert result.failed_ranges == ((2, 4),)
    assert service.payload_reads == 0  # Waiting retrieves only the manifest.
result.status, result.successful_ranges, result.failed_ranges

Close the first client and reopen the same durable job. Numerical access is explicit and binds the manifest reference. Failed positions remain null in the original query order; vectors use the pinned class order. Result accessors are immutable, while `to_dict()` is a detached copy.

In [ ]:
with notebook_batch_client(service) as client:
    reopened = client.batch_result(job_id)
    assert reopened.to_dict() == result.to_dict()
    assert service.payload_reads == 0
    payload = client.batch_payload(job_id, result_reference=reopened.result_reference)
    assert isinstance(payload, BatchPayload)
    assert len(payload.predictions) == 5
    assert payload.predictions[2:4] == (None, None)
    assert payload.probabilities[2:4] == (None, None)
    assert payload.classes == ("negative", "positive")
    assert payload.errors[0]["range"] == (2, 4)
    detached = payload.to_dict()
    detached["predictions"][0] = "changed local copy"
    assert payload.predictions[0] != "changed local copy"
    operation_id = client.job(job_id).inspect()["operation_id"]
    events = [entry for entry in client.usage() if entry["operation_id"] == operation_id]
    assert operation_id != job_id
    assert [entry["status"] for entry in events] == ["queued", "partially_completed"]
    assert all(entry["kind"] == "batch_predict" for entry in events)
len(payload.predictions), payload.classes, payload.errors[0]["range"]

The fixture now advances its synthetic server clock beyond payload retention. This is a controlled error response, not seven days of elapsed hosted acceptance. The manifest remains readable; downloading returns typed expiry. No stored object is deleted.

A real local wait timeout leaves server work running. Reconnect with the job ID. Cancellation must be requested explicitly with `client.job(job_id).cancel()`; it fences subsequent ranges and late output but does not promise to preempt an in-flight native call. This notebook does not issue a cancellation.

In [ ]:
service.expired = True
with notebook_batch_client(service) as client:
    assert client.batch_result(job_id).to_dict() == result.to_dict()
    try:
        client.batch_payload(job_id)
    except ResultExpiredError as error:
        assert error.code == "result_expired"
    else:
        raise AssertionError("Expected explicit payload expiry")
    assert client.batch_result(job_id).status == "partially_completed"
"Manifest retained; payload expired; no deletion"

Native chunk qualification, PostgreSQL concurrent claims, canonical usage, clean SDK and hosted capacity/expiry remain separate release gates. Synthetic fixture success does not close the complete API/SDK milestone. See [the batch guide](../../docs/batch-predictions.md) for async methods, bounded retries and terminal-state semantics.